# Nested Supervised Training

This notebook trains a supervised target on the bundled Breast Cancer Wisconsin dataset. The example reshapes selected columns into a nested `measurements` array so the model sees repeated measurement objects rather than one wide flat row.

The imports mirror the training tutorial. The Breast Cancer records are already buffered as nested JSONL, so the notebook can focus on the schema.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

Each record contains a list of measurement dictionaries plus a diagnosis label. This is intentionally small, but it demonstrates the pattern used for orders with line items, sessions with events, or entities with repeated attributes.


In [2]:
records = pl.read_ndjson("docs/data/breast-cancer.jsonl").head(32)

records.head()

measurements,diagnosis
list[struct[2]],str
"[{""mean_radius"",17.99}, {""mean_texture"",10.38}, … {""mean_smoothness"",0.1184}]","""malignant"""
"[{""mean_radius"",13.54}, {""mean_texture"",14.36}, … {""mean_smoothness"",0.09779}]","""benign"""
"[{""mean_radius"",20.57}, {""mean_texture"",17.77}, … {""mean_smoothness"",0.08474}]","""malignant"""
"[{""mean_radius"",13.08}, {""mean_texture"",15.71}, … {""mean_smoothness"",0.1075}]","""benign"""
"[{""mean_radius"",19.69}, {""mean_texture"",21.25}, … {""mean_smoothness"",0.1096}]","""malignant"""


The nested `Array` defines the measurement context. Inside that array, `name` identifies the measurement and `value` carries the numeric signal. The root-level `diagnosis` field is the supervised target. The field names match the record shape, so the child queries are inferred.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", max_vocab_size=16),
        j2v.Number("value"),
        name="measurements",
        max_length=8,
    ),
    j2v.Category("diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

The data module does not need special nested-data code. The schema queries describe where values live, and the data module handles batching and tensorization.

In [4]:
datamodule = j2v.PolarsDataModule(
    model=model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

Training here is intentionally minimal: the notebook proves the schema shape and supervised path, not benchmark performance.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The Rich display is useful for nested schemas because it shows which fields belong to the root record and which belong to the repeated measurement context.

In [6]:
model

Model [model] batch_size=8 d_model=16 parameters=25,031 arrays=2 fields=3 targets=1 embeds=1
`-- record [root] embed attention=mha n_layers=1 n_heads=4 n_linear=1
    |-- measurements [array] max_length=8 overflow=head attention=mha n_layers=1 n_heads=4 n_linear=1
    |   |-- name [category] active query=[*].measurements[*].name
    |   |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |   |    max_vocab_size=16 p_unavailable=0.01 topk=[]
    |   `-- value [number] active query=[*].measurements[*].value
    |        pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |        jitter=0 n_bands=8 offset=4 objective=mae
    `-- diagnosis [category] active target query=[*].diagnosis
         pooling=query weight=1 p_mask=0 p_prune=1 n_heads=4 n_linear=1
         max_vocab_size=2 p_unavailable=0.01 topk=[]

Prediction decodes only configured targets. In this case, the output is the model response for `record/diagnosis`, keyed by the same address shown in the schema display.

In [7]:
batch = records.to_dicts()[:3]
pprint(model.predict(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.2581303119659424,
│   │   │   │   -0.0766323134303093,
│   │   │   │   -0.1529366672039032,
│   │   │   │   0.14771342277526855,
│   │   │   │   0.03157772123813629,
│   │   │   │   -0.3195917308330536,
│   │   │   │   0.08580265939235687,
│   │   │   │   0.29402899742126465,
│   │   │   │   -0.2634029984474182,
│   │   │   │   -0.31875014305114746,
│   │   │   │   -0.1881491243839264,
│   │   │   │   0.4618729054927826,
│   │   │   │   0.08039034903049469,
│   │   │   │   0.5047749280929565,
│   │   │   │   -0.056441470980644226,
│   │   │   │   0.034364596009254456
│   │   │   ],
│   │   │   [
│   │   │   │   -0.25725987553596497,
│   │   │   │   -0.07663745433092117,
│   │   │   │   -0.1547514796257019,
│   │   │   │   0.14620327949523926,
│   │   │   │   0.030948519706726074,
│   │   │   │   -0.320153146982193,
│   │   │   │   0.08572274446487427,
│   │   │   │   0.29196035861968994,
│   │   │   │   -0.26374003291130066,
│   │   │   │   -0.3172081708908081,
│   │   │   │   -0.18869350850582123,
│   │   │   │   0.46309545636177063,
│   │   │   │   0.08305184543132782,
│   │   │   │   0.5050691366195679,
│   │   │   │   -0.05594658479094505,
│   │   │   │   0.0347931832075119
│   │   │   ],
│   │   │   [
│   │   │   │   -0.2584824562072754,
│   │   │   │   -0.07708965986967087,
│   │   │   │   -0.15677456557750702,
│   │   │   │   0.146638885140419,
│   │   │   │   0.03306739404797554,
│   │   │   │   -0.31829124689102173,
│   │   │   │   0.08496351540088654,
│   │   │   │   0.28785842657089233,
│   │   │   │   -0.2607018053531647,
│   │   │   │   -0.31808680295944214,
│   │   │   │   -0.19063472747802734,
│   │   │   │   0.46435028314590454,
│   │   │   │   0.08661331236362457,
│   │   │   │   0.505974292755127,
│   │   │   │   -0.05500117316842079,
│   │   │   │   0.03215910866856575
│   │   │   ]
│   │   ]
│   },
│   'record/diagnosis': {
│   │   'state': {
│   │   │   'valued': [0.565720796585083, 0.5658425688743591, 0.566077470779419],
│   │   │   'null': [0.06380893290042877, 0.06369861215353012, 0.06367845088243484],
│   │   │   'padded': [0.04830443114042282, 0.048269111663103104, 0.04817329719662666],
│   │   │   'masked': [0.2154177874326706, 0.2154412865638733, 0.21531546115875244],
│   │   │   'other': [0.10674804449081421, 0.10674852877855301, 0.1067553237080574]
│   │   },
│   │   'content': {
│   │   │   'value': ['malignant', 'malignant', 'malignant'],
│   │   │   'probability': [0.6914156675338745, 0.6905157566070557, 0.6900668144226074],
│   │   │   'topk': [[], [], []]
│   │   }
│   }
}